## Importaciones de Librerías

In [109]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


import optuna
from xgboost import XGBClassifier

import joblib

import requests

from dotenv import load_dotenv
import os

## Carga de Datos

In [110]:
load_dotenv()
URL_BACKEND_INTERNSHIPS = os.getenv('URL_BACKEND_INTERNSHIPS')

In [111]:
def get_internships():
    try:
        response = requests.get(URL_BACKEND_INTERNSHIPS)
        if response.status_code == 200:
            return response.json()
        else:
            print(f'Error al obtener los datos: {response.status_code} - {response.text}')
            return None
    except Exception as e:
        print(f'Error con la conexión a la API de las prácticas: {e}')
        return None

In [112]:
df = pd.DataFrame(get_internships())
print("Datos cargados exitosamente")

Datos cargados exitosamente


## Preparación variables X, y

In [113]:
df.columns

Index(['aptitude_score', 'github_score', 'coding_test_score',
       'soft_skills_score', 'cgpa', 'resume_score', 'interview_score', 'id',
       'extracurricular', 'consistency_score', 'skills_score', 'college_tier',
       'backlogs', 'projects_count', 'hackathons_participated',
       'placement_training', 'internships_done', 'certifications_count',
       'selected', 'communication_score', 'linkedin_activity_score'],
      dtype='str')

In [114]:
# Definimos solo las variables que demostraron verdadero valor predictivo
variables_seleccionadas = [
    'interview_score', 'skills_score', 'communication_score', 'coding_test_score',
    'projects_count', 'soft_skills_score', 'certifications_count', 'resume_score',
    'internships_done', 'extracurricular', 'consistency_score', 'placement_training',
    'cgpa', 'github_score'
]

X = df[variables_seleccionadas]
y = df['selected']

## Separación datos en entreno y test

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)

## Búsuqueda de Mejores Hiperparametros con Optuna

In [116]:
def objective_xgb(trial):
    params = {
        'max_depth' : trial.suggest_int('max_depth', 2, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'subsample': trial.suggest_float('subsample', 0.3, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'eval_metric': 'logloss',
        'random_state': 42
    }

    model = XGBClassifier(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=25)
print("XGBoost - Mejores hiperparámetros:", study_xgb.best_params)

[I 2026-06-24 11:54:03,502] A new study created in memory with name: no-name-1a0d8187-7127-411b-8a29-358ed3db2fd1
[I 2026-06-24 11:54:05,286] Trial 0 finished with value: 0.5716166568730736 and parameters: {'max_depth': 6, 'learning_rate': 0.14571831530628812, 'n_estimators': 68, 'gamma': 6.262514881256629, 'subsample': 0.9165503557478185, 'colsample_bytree': 0.5389204528526168}. Best is trial 0 with value: 0.5716166568730736.
[I 2026-06-24 11:54:12,719] Trial 1 finished with value: 0.5638255911874727 and parameters: {'max_depth': 12, 'learning_rate': 0.038651395401564544, 'n_estimators': 223, 'gamma': 1.8520090977509118, 'subsample': 0.46964654706007003, 'colsample_bytree': 0.6464086039573667}. Best is trial 0 with value: 0.5716166568730736.
[I 2026-06-24 11:54:16,636] Trial 2 finished with value: 0.5807775883133521 and parameters: {'max_depth': 2, 'learning_rate': 0.0578116784337231, 'n_estimators': 229, 'gamma': 0.5183445433945155, 'subsample': 0.7083484539108434, 'colsample_bytree'

XGBoost - Mejores hiperparámetros: {'max_depth': 5, 'learning_rate': 0.014465977633221733, 'n_estimators': 208, 'gamma': 1.5857644231935406, 'subsample': 0.4310016244027395, 'colsample_bytree': 0.5958981339921876}


## Pipeline

In [117]:
# Número de negativos / positivos en tu set de entrenamiento
num_negativos = (y_train == 0).sum()
num_positivos = (y_train == 1).sum()
ratio_peso = num_negativos / num_positivos

pipeline = Pipeline(
    steps=[
        (
            'model',
            XGBClassifier(**study_xgb.best_params,
            scale_pos_weight=ratio_peso, # Balancea el peso de las clases,
            eval_metric='logloss',
            random_state=42))
    ]
)

In [118]:
print("Entrenando el modelo final con los mejores parámetros...")
pipeline.fit(X_train, y_train)

Entrenando el modelo final con los mejores parámetros...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U20](14,)","['interview_score','skills_score','communication_score',..., 'placement_training','cgpa','github_score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


In [119]:
# Calcular predicciones para ambos conjuntos
y_train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calcular el ROC-AUC para ambos
auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print(f"ROC-AUC Entrenamiento: {auc_train:.4f}")
print(f"ROC-AUC Prueba (Test):  {auc_test:.4f}")

# Calcular la brecha (gap)
brecha = auc_train - auc_test
print(f"Brecha de Overfitting:   {brecha:.4f}")

ROC-AUC Entrenamiento: 0.7314
ROC-AUC Prueba (Test):  0.5815
Brecha de Overfitting:   0.1499


In [120]:
print("\nEvaluando el modelo en el conjunto de prueba (X_test)...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]


Evaluando el modelo en el conjunto de prueba (X_test)...


In [121]:
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score en Test: {roc_auc_score(y_test, y_pred_proba):.4f}")


=== Reporte de Clasificación ===
              precision    recall  f1-score   support

           0       0.31      0.50      0.38       525
           1       0.77      0.61      0.68      1475

    accuracy                           0.58      2000
   macro avg       0.54      0.55      0.53      2000
weighted avg       0.65      0.58      0.60      2000

=== Matriz de Confusión ===
[[261 264]
 [580 895]]

ROC-AUC Score en Test: 0.5815


## Guardado del modelo

In [122]:
joblib.dump(pipeline, '../models/xgb_model.pkl')

['../models/xgb_model.pkl']